In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path
import time

dbalchemy = create_engine(f"postgresql+psycopg2://postgres@localhost:5432/dns_mac")

In [2]:
def get(day, hour0, hour1):
    query = f"""
    WITH 
    WINDOWS AS (
        SELECT id, dn_id, is_r, rcode, macsrc, macdst, FLOOR(SECONDS/3600) as hour FROM message3_it2016_0
        WHERE FLOOR(SECONDS/3600) BETWEEN {hour0} AND {hour1}
    ),
    TMP AS (
        SELECT
        {day} AS "day",
        hour AS "hour",
        dn.dac_family_rank1 AS DAC_FAMILY,
        
        COUNT(*) AS QR,
        COUNT(*) FILTER (WHERE IS_R IS FALSE) AS Q,
        COUNT(*) FILTER (WHERE RCODE = 0) AS OK,
        COUNT(*) FILTER (WHERE RCODE = 3) AS NX,
        COUNT(*) FILTER (WHERE NOT dn.regex_check) AS QR_NOTVALID,
        COUNT(*) FILTER (WHERE IS_R IS FALSE AND NOT dn.regex_check) AS Q_NOTVALID,
        COUNT(*) FILTER (WHERE RCODE = 3 AND NOT dn.regex_check) AS NX_NOTVALID,
        COUNT(*) FILTER (WHERE RN=1) AS FA,
        COUNT(*) FILTER (WHERE RCODE=0 AND RN_QR_RCODE=1) AS FA_OK,
        COUNT(*) FILTER (WHERE RCODE=3 AND RN_QR_RCODE=1) AS FA_NX,
        COUNT(*) FILTER (WHERE RN_MAC=1) AS MACFA,
        COUNT(*) FILTER (WHERE RCODE=0 AND RN_MAC_QR_RCODE=1) AS MACFA_OK,
        COUNT(*) FILTER (WHERE RCODE=3 AND RN_MAC_QR_RCODE=1) AS MACFA_NX,


        COUNT(DISTINCT DN_ID) FILTER (WHERE P1) AS  P1_DN,
        COUNT(DISTINCT DN_ID) FILTER (WHERE NOT dn.regex_check AND P1) AS  P1_DN_NOTVALID,
        COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=0 AND P1) AS  P1_DN_OK,
        COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=3 AND P1) AS  P1_DN_NXD,
        COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=3 AND NOT dn.regex_check AND P1) AS  P1_DN_NXD_NOTVALID,
        COUNT(*) FILTER (WHERE P1) AS  P1_QR,
        COUNT(*) FILTER (WHERE IS_R IS FALSE AND P1) AS  P1_Q,
        COUNT(*) FILTER (WHERE RCODE = 0 AND P1) AS  P1_OK,
        COUNT(*) FILTER (WHERE RCODE = 3 AND P1) AS  P1_NX,
        COUNT(*) FILTER (WHERE NOT dn.regex_check AND P1) AS  P1_QR_NOTVALID,
        COUNT(*) FILTER (WHERE IS_R IS FALSE AND NOT dn.regex_check AND P1) AS  P1_Q_NOTVALID,
        COUNT(*) FILTER (WHERE RCODE = 3 AND NOT dn.regex_check AND P1) AS  P1_NX_NOTVALID,
        COUNT(*) FILTER (WHERE RN=1 AND P1) AS  P1_FA,
        COUNT(*) FILTER (WHERE RCODE=0 AND RN_QR_RCODE=1 AND P1) AS  P1_FA_OK,
        COUNT(*) FILTER (WHERE RCODE=3 AND RN_QR_RCODE=1 AND P1) AS  P1_FA_NX,
        COUNT(*) FILTER (WHERE RN_MAC=1 AND P1) AS  P1_MACFA,
        COUNT(*) FILTER (WHERE RCODE=0 AND RN_MAC_QR_RCODE=1 AND P1) AS  P1_MACFA_OK,
        COUNT(*) FILTER (WHERE RCODE=3 AND RN_MAC_QR_RCODE=1 AND P1) AS  P1_MACFA_NX,

        SUM(LOGIt1) AS LLR1_QR,
        SUM(LOGIt1) FILTER (WHERE IS_R IS FALSE) AS LLR1_Q,
        SUM(LOGIt1) FILTER (WHERE RCODE = 0) AS LLR1_OK,
        SUM(LOGIt1) FILTER (WHERE RCODE = 3) AS LLR1_NX,
        SUM(LOGIt1) FILTER (WHERE NOT dn.regex_check) AS LLR1_QR_NOTVALID,
        SUM(LOGIt1) FILTER (WHERE IS_R IS FALSE AND NOT dn.regex_check) AS LLR1_Q_NOTVALID,
        SUM(LOGIt1) FILTER (WHERE RCODE = 3 AND NOT dn.regex_check) AS LLR1_NX_NOTVALID,
        SUM(LOGIt1) FILTER (WHERE RN=1) AS LLR1_FA,
        SUM(LOGIt1) FILTER (WHERE RCODE=0 AND RN_QR_RCODE=1) AS LLR1_FA_OK,
        SUM(LOGIt1) FILTER (WHERE RCODE=3 AND RN_QR_RCODE=1) AS LLR1_FA_NX,
        SUM(LOGIt1) FILTER (WHERE RN_MAC=1) AS LLR1_MACFA,
        SUM(LOGIt1) FILTER (WHERE RCODE=0 AND RN_MAC_QR_RCODE=1) AS LLR1_MACFA_OK,
        SUM(LOGIt1) FILTER (WHERE RCODE=3 AND RN_MAC_QR_RCODE=1) AS LLR1_MACFA_NX

        FROM WINDOWS M
        JOIN bigdn_m3_3 DN ON M.DN_ID=DN.ID
        JOIN M3_RN ON M3_RN.id=M.ID
        GROUP BY "day", hour, DAC_FAMILY
    )
    SELECT tmp.* FROM tmp
    """
    dfright = pd.read_sql(query, dbalchemy)
    # if Path('/tmp/dfright.csv').exists():
    #     dfright = pd.read_csv('/tmp/dfright.csv', index_col=0)
    # else:
    #     dfright = pd.read_sql(query, dbalchemy)
    #     dfright.to_csv('/tmp/dfright.csv')
    #     pass
    dflefts = []
    for idx, row in dfright.iterrows():
        dflefts.append(pd.read_sql(f"""select * from get_hourdn_statistics('{row['dac_family']}', {row['hour']}::int)""", dbalchemy))
    return pd.concat([pd.concat(dflefts).reset_index(drop=True), dfright.drop(columns='dac_family')], axis=1)

In [46]:

hstep = 2
for day in range(10):
    dfs = []
    for h in range(0, 24 - hstep + 1, hstep):
        s = time.time()
        h0, h1 = h, h + hstep - 1
        print(day, h0, h1, end=' ')
        fn = f'hourly/day{day}_{h0}_{h1}.csv'
        if Path(fn).exists():
            df = pd.read_csv(fn, index_col=0)
        else:
            df = get(day, h, h1) # BETWEEN is INCLUSIVE
        df.to_csv(fn)
        dfs.append(df)
        print(f'{time.time() - s}s')
        pass
    pd.concat(dfs, ignore_index=True).to_csv(f'daily/day{day}.csv')
    pass


0 0 1 0.004935026168823242s
0 2 3 0.0023398399353027344s
0 4 5 0.0018770694732666016s
0 6 7 0.0018661022186279297s
0 8 9 0.0020411014556884766s
0 10 11 0.0018160343170166016s
0 12 13 0.0018601417541503906s
0 14 15 0.0017290115356445312s
0 16 17 0.0015208721160888672s
0 18 19 0.0014400482177734375s
0 20 21 0.0013720989227294922s
0 22 23 0.0016980171203613281s
1 0 1 0.0016078948974609375s
1 2 3 0.0012972354888916016s
1 4 5 0.0012280941009521484s
1 6 7 0.0013298988342285156s
1 8 9 0.0011830329895019531s
1 10 11 0.0011670589447021484s
1 12 13 0.0011110305786132812s
1 14 15 0.0010612010955810547s
1 16 17 0.0010979175567626953s
1 18 19 0.0010650157928466797s
1 20 21 0.00096893310546875s
1 22 23 0.0009710788726806641s
2 0 1 0.0010020732879638672s
2 2 3 0.0010030269622802734s
2 4 5 0.0010061264038085938s
2 6 7 0.0008919239044189453s
2 8 9 0.0008909702301025391s
2 10 11 0.0009629726409912109s
2 12 13 0.0009288787841796875s
2 14 15 0.0008807182312011719s
2 16 17 0.0008728504180908203s
2 18 19 0.

In [47]:

col_id = [
    'day',
    'hour',
    'dac_family'
]
col_dn = [
    'dn_count',
    'dn_notvalid_count',
    'dn_ok_count',
    'dn_nxd_count',
    'dn_nxd_notvalid_count'
]
col_p1_dn = [ 'p1_' + c for c in col_dn ]

col_total = [
    'qr',
    'q',
    'ok',
    'nx',
    'qr_notvalid',
    'q_notvalid',
    'nx_notvalid',
    'fa',
    'fa_ok',
    'fa_nx',
    'macfa',
    'macfa_ok',
    'macfa_nx'
]

col_p1 = [ 'p1_'+c for c in col_total]
col_llr = [ 'llr1_'+c for c in col_total]

col_malicious = col_total + col_p1 + col_llr
col_features = ['hour'] + col_dn + col_total + col_p1 + col_llr
hstep = 2
for day in range(10):
    print(day)
    df = pd.read_csv(f'daily/without_p1_dn/day{day}.csv', index_col=0)
    df = df.drop(columns='hour.1')
    joins = []
    for idxrow, row in df.iterrows():
        print('\t', idxrow)
        joins.append(pd.read_sql(f"""select * from get_hourdn_statistics('{row['dac_family']}', {row['hour']}::int)""", dbalchemy))
        pass
    df_join = pd.concat(joins)
    dfjoined = df.merge(df_join, left_on=['hour', 'dac_family'], right_on=['hour', 'dac_family'], suffixes=('','_y')).drop(columns=[c + '_y' for c in col_dn])
    dfjoined.to_csv(f'daily/day{day}.csv')
    pass

0
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
	 10
	 11
	 12
	 13
	 14
	 15
	 16
	 17
	 18
	 19
	 20
	 21
	 22
	 23
	 24
	 25
	 26
	 27
	 28
	 29
	 30
	 31
	 32
	 33
	 34
	 35
	 36
	 37
	 38
	 39
	 40
	 41
	 42
	 43
	 44
	 45
	 46
	 47
	 48
	 49
	 50
	 51
	 52
	 53
	 54
	 55
	 56
	 57
	 58
	 59
	 60
	 61
	 62
	 63
	 64
	 65
	 66
	 67
	 68
	 69
	 70
	 71
	 72
	 73
	 74
	 75
	 76
	 77
	 78
	 79
	 80
	 81
	 82
	 83
	 84
	 85
	 86
1
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
	 10
	 11
	 12
	 13
	 14
	 15
	 16
	 17
	 18
	 19
	 20
	 21
	 22
	 23
	 24
	 25
	 26
	 27
	 28
	 29
	 30
	 31
	 32
	 33
	 34
	 35
	 36
	 37
	 38
	 39
	 40
	 41
	 42
	 43
	 44
	 45
	 46
	 47
	 48
	 49
	 50
	 51
	 52
	 53
	 54
	 55
	 56
	 57
	 58
	 59
	 60
	 61
	 62
	 63
	 64
	 65
	 66
	 67
	 68
	 69
	 70
	 71
	 72
	 73
	 74
	 75
	 76
	 77
	 78
	 79
	 80
	 81
	 82
	 83
	 84
	 85
	 86
2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
	 10
	 11
	 12
	 13
	 14
	 15
	 16
	 17
	 18
	 19
	 20
	 21
	 22
	 23
	 24
	 25
	 26
	 27
	 28
	 29
	 30